<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 2


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[Описание задачи:
Создать базовый класс Student в C#, который будет представлять информацию о студентах. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;


public delegate void StudentHandler(string message);


public interface IInternship
{
    void DoInternship();
}

public class Student
{
    public string Name { get; set; }
    public int Age { get; set; }
    public int Course { get; set; }
    public string Id { get; set; }
    public double Gpa { get; set; }

    public event StudentHandler GradeReceived; 

    public Student(string name, int age, int course, string id, double gpa)
    {
        Name = name;
        Age = age;
        Course = course;
        Id = id;
        Gpa = gpa;
    }

    public virtual string GetInfo() => $"Студент: {Name}, возраст: {Age}, курс: {Course}, ID: {Id}, GPA: {Gpa}";
    public virtual void Study() => Console.WriteLine($"{Name} учится");
    public virtual void TakeExam() => Console.WriteLine($"{Name} сдаёт экзамен");

    public void TakeExam(int grade)
    {
        Console.WriteLine($"{Name} сдал экзамен на {grade}");
        GradeReceived?.Invoke($"Событие: {Name} получил оценку {grade}!");
    }
}


public class BachelorStudent : Student, IInternship
{
    public string Specialty { get; set; }
    public string InternshipCompany { get; set; }

    public BachelorStudent(string name, int age, int course, string id, double gpa, string specialty)
        : base(name, age, course, id, gpa)
    {
        Specialty = specialty;
    }

    public override void Study() => Console.WriteLine($"{Name} изучает предметы по специальности {Specialty}");

    void IInternship.DoInternship()
    {
        Console.WriteLine($"{Name} проходит стажировку в {InternshipCompany} (явная реализация)");
    }
}

public class MasterStudent : Student
{
    public string ScientificAdvisor { get; set; }

    public MasterStudent(string name, int age, int course, string id, double gpa, string advisor)
        : base(name, age, course, id, gpa)
    {
        ScientificAdvisor = advisor;
    }

    public override void TakeExam() => Console.WriteLine($"{Name} сдаёт экзамен под руководством {ScientificAdvisor}");
    
    public void Study(int hours) => Console.WriteLine($"{Name} изучает тему уже {hours} часов");
}


public class StudentService
{
    private readonly IInternship _internship;
    public StudentService(IInternship internship) { _internship = internship; }
    public void StartInternship() { _internship.DoInternship(); }
}


public class StudentCollection<T> where T : Student
{
    private List<T> _students = new List<T>();
    
    private Dictionary<string, T> _studentDict = new Dictionary<string, T>();
    
    private HashSet<string> _uniqueSpecialties = new HashSet<string>();

    public void Add(T student)
    {
        _students.Add(student);
        _studentDict[student.Id] = student; 
        
        if (student is BachelorStudent b)
        {
            _uniqueSpecialties.Add(b.Specialty);
        }
        Console.WriteLine($"Зачислен: {student.Name} (ID: {student.Id})");
    }

    public void DisplayAll()
    {
        Console.WriteLine("--- Список студентов ---");
        foreach (var s in _students)
        {
            Console.WriteLine(s.GetInfo());
        }
    }

    public void ShowUniqueSpecialties()
    {
        Console.WriteLine("--- Уникальные специальности (HashSet) ---");
        foreach (var spec in _uniqueSpecialties)
        {
            Console.WriteLine(spec);
        }
    }

    public T FindById(string id) => _studentDict.TryGetValue(id, out T student) ? student : null;
}


Console.WriteLine("=== Создание студентов ===");
BachelorStudent b = new BachelorStudent("Mike", 18, 1, "B001", 4.5, "Программирование");
b.InternshipCompany = "Яндекс";

MasterStudent m = new MasterStudent("Monica", 23, 1, "M001", 4.8, "Иванов И.И.");

b.GradeReceived += msg => Console.WriteLine(msg); 
m.GradeReceived += msg => Console.WriteLine(msg);

Console.WriteLine("\n=== Работа коллекций ===");
StudentCollection<Student> group = new StudentCollection<Student>();
group.Add(b);
group.Add(m);
group.DisplayAll();

Console.WriteLine("\n=== Поиск по Dictionary ===");
var found = group.FindById("M001");
if (found != null) Console.WriteLine($"Найден по ID: {found.GetInfo()}");

Console.WriteLine("\n=== HashSet специальностей ===");
group.ShowUniqueSpecialties();

Console.WriteLine("\n=== События и перегрузка ===");
b.TakeExam();    
b.TakeExam(5);  
m.Study(10);     

Console.WriteLine("\n=== Управление зависимостями (DI) ===");
StudentService service = new StudentService(b);
service.StartInternship();

=== Создание студентов ===

=== Работа коллекций ===
Зачислен: Mike (ID: B001)
Зачислен: Monica (ID: M001)
--- Список студентов ---
Студент: Mike, возраст: 18, курс: 1, ID: B001, GPA: 4.5
Студент: Monica, возраст: 23, курс: 1, ID: M001, GPA: 4.8

=== Поиск по Dictionary ===
Найден по ID: Студент: Monica, возраст: 23, курс: 1, ID: M001, GPA: 4.8

=== HashSet специальностей ===
--- Уникальные специальности (HashSet) ---
Программирование

=== События и перегрузка ===
Mike сдаёт экзамен
Mike сдал экзамен на 5
Событие: Mike получил оценку 5!
Monica изучает тему уже 10 часов

=== Управление зависимостями (DI) ===
Mike проходит стажировку в Яндекс (явная реализация)
